#### The below code is from the standard simple attention Mechanism

In [1]:
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={'<|endoftext|>'})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt, batch_size=4, max_length=256, stride=128, shuffle=True):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

    return dataloader


with open("/Users/anand/Desktop/LLM From Scratch/Dataset/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)

vocab_size = 50257
output_dim = 256
max_len = 1024
context_length = max_len


token_embedding_layer = nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

max_length = 4
dataloader = create_dataloader(raw_text, batch_size=8, max_length=max_length, stride=max_length)

In [2]:
for batch in dataloader:
    x, y = batch

    token_embeddings = token_embedding_layer(x)
    pos_embeddings = pos_embedding_layer(torch.arange(max_length))

    input_embeddings = token_embeddings + pos_embeddings

    break

## 🧠 Multi-Query Attention (MQA) – Code Walkthrough

Below is a **step-by-step explanation of the PyTorch implementation of Multi-Query Attention (MQA)**.  
The goal is to clearly understand:

- What each line of code does
- How tensors change shape
- How attention is computed
- How MQA differs from standard Multi-Head Attention

---

### ⚙️ Step 1 — Initialization (`__init__`)

The constructor sets up **all attention layers and parameters**.

### 🔢 Input Arguments

| Parameter        | Meaning                    |
| ---------------- | -------------------------- |
| `d_in`           | Input embedding dimension  |
| `d_out`          | Output embedding dimension |
| `context_length` | Maximum sequence length    |
| `dropout`        | Dropout probability        |
| `num_heads`      | Number of attention heads  |

---

### 🧩 Head Dimension

```python
self.head_dim = d_out // num_heads
```

This computes the **dimension per head**.

$$
D = \frac{d_{out}}{H}
$$

Where:

* $D$ = head dimension
* $H$ = number of heads

Example:

```
d_out = 768
num_heads = 12
```

$$
D = 768 / 12 = 64
$$

So each attention head processes **64 features**.

---

### 🔍 Step 2 — Query, Key, Value Projections

### Query Projection

```python
self.W_query = nn.Linear(d_in, d_out)
```

This produces **multiple query heads**.

Mathematically:

$$
Q = XW_Q
$$

Where:

$$
W_Q \in \mathbb{R}^{d_{in} \times d_{out}}
$$

Output shape:

```
(B,T,d_out)
```

---

### Key Projection (Shared)

```python
self.W_key = nn.Linear(d_in, head_dim)
```

In **MQA we create only ONE key head**.

$$
K = XW_K
$$

Where:

$$
W_K \in \mathbb{R}^{d_{in} \times D}
$$

Output shape:

```
(B,T,D)
```

---

### Value Projection (Shared)

```python
self.W_value = nn.Linear(d_in, head_dim)
```

$$
V = XW_V
$$

Where:

$$
W_V \in \mathbb{R}^{d_{in} \times D}
$$

Output shape:

```
(B,T,D)
```

---

### 🧠 Step 3 — Causal Mask

```python
self.register_buffer(
    "mask",
    torch.triu(torch.ones(context_length, context_length), diagonal=1)
)
```

This creates a **causal mask**.

Example mask:

```
0 1 1 1
0 0 1 1
0 0 0 1
0 0 0 0
```

Meaning:

* A token **cannot attend to future tokens**

Used in **autoregressive models like GPT**.

---

### 📥 Step 4 — Input Tensor

Inside the forward pass:

```python
b, num_tokens, d_in = x.shape
```

Input shape:

$$
X \in \mathbb{R}^{B \times T \times C}
$$

Where:

| Symbol | Meaning             |
| ------ | ------------------- |
| $B$    | Batch size          |
| $T$    | Number of tokens    |
| $C$    | Embedding dimension |

Example:

```
(2,1024,768)
```

---

### 🔧 Step 5 — Compute Q, K, V

```python
queries = self.W_query(x)
keys = self.W_key(x)
values = self.W_value(x)
```

Shapes become:

```
queries → (B,T,d_out)
keys    → (B,T,D)
values  → (B,T,D)
```

---

### 🧩 Step 6 — Split Queries into Heads

```python
queries = queries.view(b, num_tokens, num_heads, head_dim)
queries = queries.transpose(1,2)
```

Shape transformation:

```
(B,T,C)
↓
(B,T,H,D)
↓
(B,H,T,D)
```

This allows each head to **learn different relationships**.

---

### 🔄 Step 7 — Expand Shared Keys and Values

Because MQA uses **one KV head**, we expand them.

### Add head dimension

```python
keys = keys.unsqueeze(1)
values = values.unsqueeze(1)
```

Shape:

```
(B,1,T,D)
```

---

### Broadcast to all heads

```python
keys = keys.expand(b, num_heads, num_tokens, head_dim)
values = values.expand(b, num_heads, num_tokens, head_dim)
```

Shape:

```
(B,H,T,D)
```

⚠️ Important:

This **does not copy memory**.
It only creates a **view of the same tensor**.

---

### ⚙️ Step 8 — Compute Attention Scores

```python
attn_scores = queries @ keys.transpose(2,3)
```

Matrix multiplication:

$$
S = QK^T
$$

Shapes:

$$
(B,H,T,D) \times (B,H,D,T)
$$

Result:

$$
S \in \mathbb{R}^{B \times H \times T \times T}
$$

Each element represents **how strongly one token attends to another**.

---

### 🚫 Step 9 — Apply Causal Mask

```python
attn_scores.masked_fill_(mask_bool, -torch.inf)
```

Future tokens are blocked.

Example:

```
Token 3 cannot see token 4
```

Scores become:

```
-∞
```

After softmax this becomes:

```
0 attention
```

---

### ⚖️ Step 10 — Scale Attention

```python
attn_scores / (self.head_dim ** 0.5)
```

Formula:

$$
S = \frac{QK^T}{\sqrt{D}}
$$

Why scaling?

Without scaling:

* Dot products become **too large**
* Softmax becomes unstable

---

### 🔄 Step 11 — Softmax

```python
attn_weights = torch.softmax(attn_scores, dim=-1)
```

Formula:

$$
A = softmax(S)
$$

Shape:

```
(B,H,T,T)
```

Each row becomes **probability distribution**.

---

### 🎯 Step 12 — Apply Attention to Values

```python
context_vec = attn_weights @ values
```

Formula:

$$
O = AV
$$

Matrix multiplication:

$$
(B,H,T,T) \times (B,H,T,D)
$$

Result:

```
(B,H,T,D)
```

Now each token contains **weighted information from other tokens**.

---

### 🔗 Step 13 — Merge Heads

```python
context_vec = context_vec.transpose(1,2)
context_vec = context_vec.view(b,num_tokens,d_out)
```

Transformation:

```
(B,H,T,D)
↓
(B,T,H,D)
↓
(B,T,C)
```

Where:

$$
C = H \times D
$$

---

### 🔧 Step 14 — Final Projection

```python
context_vec = self.out_proj(context_vec)
```

Final linear transformation:

$$
Y = OW_O
$$

Where:

$$
W_O \in \mathbb{R}^{C \times C}
$$

Output shape:

```
(B,T,C)
```

---

### 🚀 Why Multi-Query Attention is Efficient

Standard Multi-Head Attention stores **separate keys and values per head**.

Memory:

$$
K,V \in \mathbb{R}^{B \times H \times T \times D}
$$

MQA stores:

$$
K,V \in \mathbb{R}^{B \times T \times D}
$$

Memory reduction:

$$
\approx H \times
$$

Example:

```
H = 32
```

MQA reduces KV cache **32×**.

---

#### 🧠 Intuition

Think of it like:

### Multi-Head Attention

```
32 heads → 32 memories
```

### Multi-Query Attention

```
32 heads → 1 shared memory
```

All heads **query the same information store**.

---

### 🏁 Summary

Multi-Query Attention improves Transformer efficiency by:

✅ Sharing Keys and Values across heads
✅ Reducing KV cache memory
✅ Speeding up inference
✅ Scaling better for long contexts

Because of this, MQA is widely used in modern large language models.

---


In [4]:
class MultiQueryAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        # Query projection (multiple heads)
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Key / Value projection (single head only)
        self.W_key = nn.Linear(d_in, self.head_dim, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, self.head_dim, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)

        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):

        b, num_tokens, d_in = x.shape

        # projections
        queries = self.W_query(x)      # (b, num_tokens, d_out)
        keys = self.W_key(x)           # (b, num_tokens, head_dim)
        values = self.W_value(x)       # (b, num_tokens, head_dim)

        # ---- reshape queries into heads ----
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # (b, num_heads, num_tokens, head_dim)
        queries = queries.transpose(1, 2)

        # ---- expand shared K,V to all heads ----
        keys = keys.unsqueeze(1)        # (b,1,num_tokens,head_dim)
        values = values.unsqueeze(1)

        keys = keys.expand(b, self.num_heads, num_tokens, self.head_dim)
        values = values.expand(b, self.num_heads, num_tokens, self.head_dim)

        # ---- attention scores ----
        attn_scores = queries @ keys.transpose(2, 3)

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # ---- scaling + softmax ----
        attn_weights = torch.softmax(
            attn_scores / (self.head_dim ** 0.5),
            dim=-1
        )

        attn_weights = self.dropout(attn_weights)

        # ---- weighted values ----
        context_vec = (attn_weights @ values).transpose(1, 2)

        # ---- merge heads ----
        context_vec = context_vec.contiguous().view(
            b,
            num_tokens,
            self.d_out
        )

        context_vec = self.out_proj(context_vec)

        return context_vec

## 🧠 How Much KV Cache Memory Does Multi-Query Attention Reduce?

The **KV cache memory reduction** depends mainly on the **number of attention heads (H)**.

In standard **Multi-Head Attention (MHA)**, each head stores its own **Key and Value** tensors.

In **Multi-Query Attention (MQA)**, all heads **share one Key and Value**.

---

### 📦 KV Cache in Multi-Head Attention

For MHA:

$$
K,V \in \mathbb{R}^{B \times H \times T \times D}
$$

Total memory:

$$
Memory_{MHA} = 2 \times B \times H \times T \times D
$$

Where:

| Symbol | Meaning         |
| ------ | --------------- |
| $B$    | Batch size      |
| $H$    | Number of heads |
| $T$    | Sequence length |
| $D$    | Head dimension  |

The **2** comes from storing both **Keys and Values**.

---

### 📦 KV Cache in Multi-Query Attention

In MQA, keys and values are shared.

$$
K,V \in \mathbb{R}^{B \times T \times D}
$$

Total memory:

$$
Memory_{MQA} = 2 \times B \times T \times D
$$

---

### 📉 Memory Reduction Formula

Reduction factor:

$$
Reduction = \frac{Memory_{MQA}}{Memory_{MHA}}
$$

Substitute the formulas:

$$
Reduction = \frac{2BTD}{2BHTD}
$$

Cancel common terms:

$$
Reduction = \frac{1}{H}
$$

---

### 📊 Percentage Memory Reduction

The percentage reduction becomes:

$$
Reduction% = \left(1 - \frac{1}{H}\right) \times 100
$$

---

### 🔢 Example Calculations

### Example 1 — 8 Heads

$$
Reduction = (1 - 1/8) \times 100
$$

$$
Reduction = 87.5%
$$

---

### Example 2 — 16 Heads

$$
Reduction = (1 - 1/16) \times 100
$$

$$
Reduction = 93.75%
$$

---

### Example 3 — 32 Heads (common in LLMs)

$$
Reduction = (1 - 1/32) \times 100
$$

$$
Reduction = 96.875%
$$

So **MQA removes about 97% of KV cache memory**.

---

### 🧮 Real LLM Example

Typical LLM configuration:

```
Heads = 32
Head dimension = 128
Sequence length = 8192
```

### Multi-Head Attention KV

```
K,V = (B, 32, 8192, 128)
```

### Multi-Query Attention KV

```
K,V = (B, 8192, 128)
```

Memory reduction:

```
32× smaller
```

---

### 📊 Summary Table

| Heads | KV Memory Reduction |
| ----- | ------------------- |
| 8     | 87.5%               |
| 16    | 93.75%              |
| 32    | 96.875%             |
| 64    | 98.4375%            |

So as the **number of heads increases**, the **memory savings become even larger**.

---

### 🧠 Intuition

Imagine **32 attention heads**.

### Multi-Head Attention

```
Head1 → own KV memory
Head2 → own KV memory
Head3 → own KV memory
...
Head32 → own KV memory
```

Total = **32 KV memories**

---

### Multi-Query Attention

```
Head1 → shared KV
Head2 → shared KV
Head3 → shared KV
...
Head32 → shared KV
```

Total = **1 KV memory**

---

✅ **Final takeaway**

For a model with **32 heads**, Multi-Query Attention reduces KV cache memory by about:

**≈ 96.9%**

which is why it is widely used in modern LLMs for **efficient inference and long context support**.


### 🧠 Multi-Query Attention (MQA) — Advantages, Disadvantages, and Key Concepts

Multi-Query Attention (MQA) is an optimization of the Transformer attention mechanism where **multiple query heads share a single set of keys and values**.  
It is mainly designed to **reduce memory usage and improve inference speed**, especially for large language models.

---

### 🚀 Advantages of Multi-Query Attention

### 💾 1. Huge Reduction in KV Cache Memory

In standard Multi-Head Attention (MHA), every head stores its own keys and values.

### Multi-Head Attention KV Cache

$$
K,V \in \mathbb{R}^{B \times H \times T \times D}
$$

Memory requirement:

$$
2 \times B \times H \times T \times D
$$

Where:

| Symbol | Meaning |
|---|---|
| $B$ | Batch size |
| $H$ | Number of heads |
| $T$ | Sequence length |
| $D$ | Head dimension |

---

### Multi-Query Attention KV Cache

In MQA, keys and values are **shared across heads**.

$$
K,V \in \mathbb{R}^{B \times T \times D}
$$

Memory requirement:

$$
2 \times B \times T \times D
$$

---

### 📊 Memory Reduction

$$
\text{Reduction} \approx H \times
$$

Example:

```

Heads = 32

```

Memory becomes **32× smaller**.

This is one of the **biggest advantages** for long-context models.

---

### ⚡ 2. Faster Inference

Since fewer keys and values are stored:

- Less **GPU memory access**
- Smaller **KV cache**
- Faster **attention computation**

This is especially important for **autoregressive generation** where tokens are generated one by one.

---

### 🧮 3. Lower Bandwidth Requirements

During inference, the model repeatedly reads from KV cache.

In MHA:

```

Memory read = H × larger

```

In MQA:

```

Memory read = 1 × shared KV

```

This reduces **GPU memory bandwidth usage**, which is often the **real bottleneck**.

---

### 📈 4. Scales Better for Long Context

For large sequence lengths:

$$
T = 32k,\ 64k,\ 128k
$$

KV cache becomes extremely large in MHA.

MQA allows models to handle **longer contexts efficiently**.

---

### 🧠 5. Works Well with Flash Attention

MQA integrates well with **optimized attention kernels** like Flash Attention because:

- KV tensors are smaller
- Less memory movement
- Faster GPU kernels

---

#### ⚠️ Disadvantages of Multi-Query Attention

### 🧩 1. Reduced Representation Diversity

In standard Multi-Head Attention:

```

Each head learns its own keys and values

```

In MQA:

```

All heads share the same keys and values

```

This reduces **model expressiveness**.

Mathematically:

Standard MHA:

$$
K_h = XW_{K_h}
$$

MQA:

$$
K = XW_K
$$

So all heads rely on the **same key representation**.

---

### 🎯 2. Potential Loss in Model Quality

Because heads share keys and values:

- Some **fine-grained relationships may be lost**
- The model may learn **less diverse attention patterns**

In practice, this loss is **usually small**, but it can appear in certain tasks.

---

### 🔧 3. Harder to Extend Some Attention Variants

Certain attention mechanisms assume **per-head KV representations**.

Examples:

- specialized routing heads
- sparse head architectures

These become harder with shared KV.

---

### 🧠 Important Concepts Related to MQA

#### 🔑 1. KV Cache

KV cache stores keys and values from previous tokens during inference.

For autoregressive generation:

```

token 1 → compute KV
token 2 → reuse KV
token 3 → reuse KV

```

Instead of recomputing attention for all tokens.

---

### KV Cache Equation

At timestep $t$:

$$
K_{cache} = [K_1, K_2, ..., K_t]
$$

$$
V_{cache} = [V_1, V_2, ..., V_t]
$$

This allows the model to compute attention only with **new tokens**.

---

### 🧠 2. Query Heads

Queries still have multiple heads in MQA.

Query tensor:

$$
Q \in \mathbb{R}^{B \times H \times T \times D}
$$

Each head can learn **different attention patterns**, even though keys and values are shared.

---

### 🔄 3. Broadcasting

MQA uses **tensor broadcasting** to share keys and values.

Original KV:

```

(B,T,D)

```

Expanded:

```

(B,1,T,D)

```

Broadcasted:

```

(B,H,T,D)

```

This does **not copy memory**, making it efficient.

---

### 🧠 4. Grouped Query Attention (GQA)

Many modern models use **Grouped Query Attention**, which is a compromise between MHA and MQA.

Instead of:

```

32 query heads
1 KV head

```

We use:

```

32 query heads
8 KV heads

```

Formula:

$$
KV_{groups} = G
$$

Each group shares keys and values.

This improves **model quality while still saving memory**.

---

#### 📊 Comparison

| Feature | Multi-Head Attention | Multi-Query Attention |
|---|---|---|
Query Heads | H | H |
Key Heads | H | 1 |
Value Heads | H | 1 |
KV Cache Size | Large | Small |
Inference Speed | Slower | Faster |
Memory Usage | High | Low |
Model Expressiveness | Higher | Slightly Lower |

---

#### 🧠 Intuition

Think of attention like **people asking questions in a library**.

### Multi-Head Attention

```

Each person has their own library

```
```

Head1 → library1
Head2 → library2
Head3 → library3

```

---

### Multi-Query Attention

```

Everyone shares one library

```
```

Head1 → library
Head2 → library
Head3 → library

```

They ask **different questions**, but read from the **same knowledge source**.

---

### 🏁 Summary

Multi-Query Attention is an efficient attention mechanism designed for **large-scale Transformer models**.

### Key benefits

✅ Dramatically reduces KV cache memory  
✅ Improves inference speed  
✅ Scales better for long context  
✅ Works well with optimized GPU kernels  

### Trade-off

⚠️ Slight reduction in attention diversity.

Because of its efficiency, MQA (or its variant **Grouped Query Attention**) has become a **standard design choice in modern large language models**.

---
